In [1]:
##### Calculates final sub-national capital and labor intensities using final production and capital/labor rasters (after re-scaling)

from pathlib import Path
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
from pyproj import Transformer
import matplotlib.patches as mpatches
from matplotlib.colors import to_hex, Normalize
from matplotlib import cm
import matplotlib
from matplotlib.patches import Rectangle
from rasterstats import zonal_stats

In [2]:
##### Paths
cd = Path.cwd().parent.parent

capital_path    = f"{cd}/Results/Raster_model/rescaled_capital_USD.tif"
labor_path      = f"{cd}/Results/Raster_model/rescaled_jobs.tif"
production_path = f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif"

sub_national_capital_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_capital.shp")
sub_national_labor_geo   = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_labor.shp")

save_path_capital = f"{cd}/Results/Raster_model/sub_national_intensities/capital_intensity_subnational.csv"
save_path_labor = f"{cd}/Results/Raster_model/sub_national_intensities/labor_intensity_subnational.csv"

In [3]:
##### Helpers

def raster_crs(path):
    """Read just the CRS/metadata header, not the pixel data."""
    with rasterio.open(path) as src:
        return src.crs

def add_zonal_sum(gdf, raster_path, out_col, nodata):
    """
    Reprojects a *copy* of gdf to the raster's native CRS (cheap — vector only),
    then streams windowed reads off disk for the sum. Never loads the full raster
    into memory and never resamples/reprojects the raster itself.
    """
    gdf_native = gdf.to_crs(raster_crs(raster_path))
    stats = zonal_stats(
        gdf_native,
        raster_path,
        stats=["sum"],
        nodata=nodata,
        all_touched=False  # set True if you want to include pixels only partially inside a polygon
    )
    gdf[out_col] = [s["sum"] for s in stats]
    return gdf

def compute_intensity(numerator_col, denominator_col):
    """
    Divide numerator by denominator, except:
    - if either numerator or denominator is 0 -> intensity is 0
    - if denominator is NaN (i.e. no valid pixels found in the zone at all) -> intensity is NaN
    """
    numerator = numerator_col
    denominator = denominator_col
    intensity = numerator / denominator
    zero_mask = (numerator == 0) | (denominator == 0)
    intensity = intensity.where(~zero_mask, 0) if hasattr(intensity, "where") else np.where(zero_mask, 0, intensity)
    return intensity

##### Capital intensity: sum capital and production within each capital sub-national region
sub_national_capital_geo = add_zonal_sum(
    sub_national_capital_geo, capital_path, "capital_sum_USD", nodata=np.nan
)
sub_national_capital_geo = add_zonal_sum(
    sub_national_capital_geo, production_path, "production_sum_tonnes", nodata=-9999
)
sub_national_capital_geo["capital_intensity_USD_per_tonne"] = compute_intensity(
    sub_national_capital_geo["capital_sum_USD"],
    sub_national_capital_geo["production_sum_tonnes"]
)

##### Labor intensity: sum labor and production within each labor sub-national region
sub_national_labor_geo = add_zonal_sum(
    sub_national_labor_geo, labor_path, "labor_sum_jobs", nodata=np.nan
)
sub_national_labor_geo = add_zonal_sum(
    sub_national_labor_geo, production_path, "production_sum_tonnes", nodata=-9999
)
sub_national_labor_geo["labor_intensity_jobs_per_tonne"] = compute_intensity(
    sub_national_labor_geo["labor_sum_jobs"],
    sub_national_labor_geo["production_sum_tonnes"]
)

In [6]:
##### Filter columns 

sub_national_capital_geo = sub_national_capital_geo.rename(columns={"capital_intensity_USD_per_tonne": "modelled_capital_intensity_USD_per_tonne"})
sub_national_labor_geo = sub_national_labor_geo.rename(columns={"labor_intensity_jobs_per_tonne": "modelled_labor_intensity_jobs_per_tonne"})

final_capital = sub_national_capital_geo[['PROJ_ID', 'modelled_capital_intensity_USD_per_tonne']]
final_labor = sub_national_labor_geo[['PROJ_ID', 'modelled_labor_intensity_jobs_per_tonne']]

In [8]:
##### Save
final_capital.to_csv(save_path_capital, index=False)
final_labor.to_csv(save_path_labor, index=False)